# 02 - Train + Evaluate (reproducible, end to end)

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/cesaredellacorte-bot/Architectural-items_detection/blob/main/notebooks/02_train_eval.ipynb)

**How to run:** Runtime -> Change runtime type -> **T4 GPU**, then **Runtime -> Restart runtime -> Run all**. Nothing to edit, no API key, no login.

What this notebook does, in order:
1. records every parameter in one cell,
2. installs the pinned versions from `requirements.txt`,
3. downloads the **frozen dataset from the GitHub Release `dataset-v2`** and checks its **SHA256** (it stops if the file has changed),
4. cleans the labels (`Furnitures` -> `Furniture`, polygons -> boxes, duplicates removed),
5. trains YOLOv8n, 6. writes the metrics table, 7. saves the evidence images, 8. saves the FP/FN images for the error analysis,
9. writes `results/run_info.md` + `results/params.yaml` (versions, parameters, hashes) and 10. zips `results/` for download.

Runtime on a T4: about 2 min training + 2-4 min setup. No GPU? Set `FAST_VERIFY = True` (5 epochs, then evaluates the released `best.pt`).

In [ ]:
# 1. Parameters -- the ONLY cell you would ever edit (everything below reads from here)
GITHUB_USER_REPO = "cesaredellacorte-bot/Architectural-items_detection"
BRANCH           = "main"

# Frozen dataset: Roboflow export (YOLOv8 format) of project architectural_items_detection_v1, VERSION 2,
# published unchanged as a GitHub Release asset. The SHA256 below is checked before training.
DATASET_URL    = "https://github.com/cesaredellacorte-bot/Architectural-items_detection/releases/download/dataset-v2/architectural_items_v2_yolov8.zip"
DATASET_SHA256 = "7e70aedcd900e5e7c4e0b7993dc7d33c736a4eaeeb239296382a768afdb2cb48"
GDRIVE_FILE_ID = "1iVq3qa_xrTHb422iZmxqrK1AS8E2h0La"   # backup copy, used only if the Release download fails
ROBOFLOW_WORKSPACE, ROBOFLOW_PROJECT, ROBOFLOW_VERSION = "cesare-della-corte", "architectural_items_detection_v1", 2
CLASS_MERGE = {"Furnitures": "Furniture"}             # duplicate class fixed during cleaning

# Training
MODEL = "yolov8n.pt"
EPOCHS, BATCH, IMGSZ, SEED = 50, 16, 640, 0
# Evaluation / evidence
CONF, MATCH_IOU = 0.25, 0.5                            # threshold for evidence images and FP/FN matching

# Released weights (used when FAST_VERIFY = True)
WEIGHTS_URL    = "https://github.com/cesaredellacorte-bot/Architectural-items_detection/releases/download/v1.0/best.pt"
WEIGHTS_SHA256 = "65b7d90e13729e141e74d8871b8139fb27b51d93822a82c87e24a27876a3b571"
FAST_VERIFY    = False                                 # True = 5-epoch smoke run, metrics computed with released best.pt

WORK           = "/content"                            # Colab working folder
NEW_IMAGES_DIR = f"{WORK}/repo/new_images"             # floor plans NOT in the dataset (no labels)

In [ ]:
# 2. Clone repo + install pinned dependencies
import os, sys, time, hashlib
T0 = time.time()
os.makedirs(WORK, exist_ok=True)
if not os.path.exists(f"{WORK}/repo"):
    !git clone -q -b {BRANCH} https://github.com/{GITHUB_USER_REPO} {WORK}/repo
%cd {WORK}/repo
!pip install -q -r requirements.txt
!git rev-parse HEAD > /tmp/commit.txt
COMMIT = open("/tmp/commit.txt").read().strip()
sys.path.insert(0, f"{WORK}/repo/scripts")
import evidence, prepare_dataset
import ultralytics; ultralytics.checks()
print("Repo commit:", COMMIT)

In [ ]:
# 3. Download the frozen dataset, verify SHA256, extract, clean
import glob, shutil, zipfile, urllib.request

def sha256(path, chunk=1 << 20):
    h = hashlib.sha256()
    with open(path, "rb") as f:
        for b in iter(lambda: f.read(chunk), b""):
            h.update(b)
    return h.hexdigest()

ZIP = f"{WORK}/dataset.zip"
DATASET_SOURCE = "github-release"
try:
    if os.path.exists(DATASET_URL):            # local file (offline testing)
        shutil.copy(DATASET_URL, ZIP)
    else:
        urllib.request.urlretrieve(DATASET_URL, ZIP)
except Exception as e:
    print("[!] Release download failed:", e, "-> using the Google Drive backup copy")
    import gdown
    gdown.download(f"https://drive.google.com/uc?id={GDRIVE_FILE_ID}", ZIP, quiet=False)
    DATASET_SOURCE = "google-drive-backup"

ZIP_SHA = sha256(ZIP)
print(f"SHA256 {ZIP_SHA}  ({os.path.getsize(ZIP)/1e6:.2f} MB, source: {DATASET_SOURCE})")
if DATASET_SOURCE == "github-release":
    assert ZIP_SHA == DATASET_SHA256, "Dataset zip does not match the frozen SHA256 - do not train on it."
elif ZIP_SHA != DATASET_SHA256:
    print("[!] WARNING: the backup copy differs from the frozen release; results may not match the README.")

ROOT = f"{WORK}/dataset"
shutil.rmtree(ROOT, ignore_errors=True)
zipfile.ZipFile(ZIP).extractall(ROOT)
ROOT = os.path.dirname(glob.glob(f"{ROOT}/**/data.yaml", recursive=True)[0])
for s in ("train", "valid", "test"):
    print(f"{s:5s}: {len(glob.glob(f'{ROOT}/{s}/images/*'))} images")

# Merge 'Furnitures' into 'Furniture' (Walls is kept), convert polygon rows to boxes, drop duplicate boxes
DATA_YAML, CLASS_NAMES = prepare_dataset.prepare(ROOT, merge=CLASS_MERGE)
print(open(DATA_YAML).read())

In [ ]:
# 4. Train (or 5-epoch smoke run + released weights)
from ultralytics import YOLO
model = YOLO(MODEL)
model.train(data=DATA_YAML, epochs=5 if FAST_VERIFY else EPOCHS, batch=BATCH, imgsz=IMGSZ, seed=SEED,
            deterministic=True, project=f"{WORK}/runs", name="train", exist_ok=True, plots=True)
RUN_DIR = str(model.trainer.save_dir)
BEST = f"{RUN_DIR}/weights/best.pt"
if FAST_VERIFY:
    BEST = f"{WORK}/best_release.pt"
    urllib.request.urlretrieve(WEIGHTS_URL, BEST)
    assert sha256(BEST) == WEIGHTS_SHA256, "Released weights do not match the recorded SHA256"
BEST_SHA = sha256(BEST)
print("Using weights:", BEST, BEST_SHA)

In [ ]:
# 5. Metrics table (P / R / mAP50 / mAP50-95, overall + per class) + training curves
best = YOLO(BEST)
rows = evidence.metrics_table(best, DATA_YAML, split="val", imgsz=IMGSZ, out_dir="results")
evidence.copy_curves(RUN_DIR, "results/curves")
from IPython.display import Image, display
display(Image("results/curves/results.png", width=900))
display(Image("results/curves/confusion_matrix.png", width=600))

In [ ]:
# 6. Evidence pack: 5 annotation screenshots, 10 validation predictions, 5 predictions on new images
names = best.names
shutil.rmtree("results/evidence/annotations", ignore_errors=True)
shutil.rmtree("results/evidence/val_predictions", ignore_errors=True)
shutil.rmtree("results/evidence/new_predictions", ignore_errors=True)
evidence.save_annotation_examples(f"{ROOT}/train/images", f"{ROOT}/train/labels", names, n=5,
                                  out_dir="results/evidence/annotations")
evidence.save_predictions(best, f"{ROOT}/valid/images", 10, "results/evidence/val_predictions", conf=CONF, imgsz=IMGSZ)
evidence.save_predictions(best, NEW_IMAGES_DIR, 5, "results/evidence/new_predictions", conf=CONF, imgsz=IMGSZ)
for p in sorted(glob.glob("results/evidence/val_predictions/*"))[:3]:
    display(Image(p, width=600))

In [ ]:
# 7. Error-analysis images: RED = false positive, BLUE = false negative (used in docs/error_analysis.md)
shutil.rmtree("results/evidence/errors", ignore_errors=True)
errs = evidence.find_errors(best, f"{ROOT}/valid/images", f"{ROOT}/valid/labels", names, conf=CONF,
                            imgsz=IMGSZ, iou_thr=MATCH_IOU, out_dir="results/evidence/errors")
for p in sorted(glob.glob("results/evidence/errors/*"))[:6]:
    print(p); display(Image(p, width=700))

In [ ]:
# 8. Reproducibility record: versions + parameters + hashes
import yaml, platform, torch
runtime_min = round((time.time() - T0) / 60, 1)
evidence.write_run_info("results/run_info.md", extra={
    "Repo commit": COMMIT,
    "Dataset": f"{ROBOFLOW_WORKSPACE}/{ROBOFLOW_PROJECT} v{ROBOFLOW_VERSION} ({DATASET_SOURCE})",
    "Dataset SHA256": ZIP_SHA, "Weights SHA256": BEST_SHA, "Model": MODEL,
    "Epochs/Batch/Imgsz/Seed": f"{5 if FAST_VERIFY else EPOCHS}/{BATCH}/{IMGSZ}/{SEED}",
    "Mode": "verification run + released weights" if FAST_VERIFY else "full training",
    "Total runtime (min)": runtime_min})
params = {
    "dataset": {"url": DATASET_URL, "sha256_expected": DATASET_SHA256, "sha256_measured": ZIP_SHA,
                "source_used": DATASET_SOURCE, "roboflow": f"{ROBOFLOW_WORKSPACE}/{ROBOFLOW_PROJECT}/{ROBOFLOW_VERSION}",
                "class_merge": CLASS_MERGE, "classes": list(CLASS_NAMES)},
    "train": {"model": MODEL, "epochs": 5 if FAST_VERIFY else EPOCHS, "batch": BATCH, "imgsz": IMGSZ, "seed": SEED,
              "fast_verify": FAST_VERIFY},
    "eval": {"split": "val", "conf": CONF, "match_iou": MATCH_IOU},
    "weights": {"path_used": BEST, "sha256": BEST_SHA, "release_url": WEIGHTS_URL},
    "env": {"python": platform.python_version(), "ultralytics": str(ultralytics.__version__), "torch": str(torch.__version__),
            "gpu": torch.cuda.get_device_name(0) if torch.cuda.is_available() else "CPU", "repo_commit": COMMIT},
    "runtime_min": runtime_min,
}
yaml.safe_dump(params, open("results/params.yaml", "w"), sort_keys=False)
print(open("results/params.yaml").read())

In [ ]:
# 9. Zip results/ so you can commit them (GitHub -> Add file -> Upload files)
shutil.make_archive(f"{WORK}/results_to_commit", "zip", ".", "results")
try:
    from google.colab import files
    files.download(f"{WORK}/results_to_commit.zip")
except ImportError:
    print("Not on Colab - zip saved at", f"{WORK}/results_to_commit.zip")
print("DONE - end-to-end run finished in", round((time.time() - T0) / 60, 1), "min")